# 🤖 GenAI-Based Customer Support Chatbot V3
**Built with DialoGPT + Smart Response Engine**

**Author:** Syed Mohammad Shahzeb
**Skills:** Python, Hugging Face, Transformers, Prompt Engineering, Gradio

In [1]:
!pip install transformers gradio torch --quiet
print('✅ All libraries installed!')

✅ All libraries installed!


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr
print('✅ Libraries imported!')

✅ Libraries imported!


In [3]:
MODEL_NAME = 'microsoft/DialoGPT-medium'
print(f'⏳ Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'✅ Model loaded on: {device}')

⏳ Loading microsoft/DialoGPT-medium...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded on: cuda


In [4]:
# Smart Customer Support Response Engine
SUPPORT_RESPONSES = {
    'order': 'I can help with your order! Please share your order number.',
    'return': 'Happy to help with a return! Our policy allows returns within 30 days. Please share your order number.',
    'refund': 'Refunds are processed within 5-7 business days. Please share your order number to proceed.',
    'track': 'To track your shipment, please share your order number.',
    'payment': 'Sorry about the payment issue. Please check your card details or try another payment method.',
    'cancel': 'Orders can be cancelled within 24 hours of placement. What is your order number?',
    'delivery': 'Standard delivery takes 3-5 business days. Express takes 1-2 business days.',
    'address': 'Address changes are possible before the order is shipped. Please provide your order number.',
    'discount': 'Use code SAVE10 for 10% off your next purchase!',
    'hello': 'Hello! Welcome to Customer Support. How can I assist you today?',
    'hi': 'Hi there! Welcome to Customer Support. How can I help you today?',
    'help': 'I can help with:\n- Order tracking\n- Returns and Refunds\n- Payment issues\n- Delivery queries',
    'thank': 'You are welcome! Is there anything else I can help with?',
    'bye': 'Thank you for contacting us. Have a great day!'
}

def get_smart_response(user_message, chat_history_ids=None):
    msg_lower = user_message.lower()
    for keyword, response in SUPPORT_RESPONSES.items():
        if keyword in msg_lower:
            return response, chat_history_ids
    new_input_ids = tokenizer.encode(user_message + tokenizer.eos_token, return_tensors='pt').to(device)
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids
    if bot_input_ids.shape[-1] > 800:
        bot_input_ids = bot_input_ids[:, -800:]
    with torch.no_grad():
        new_history_ids = model.generate(bot_input_ids, max_new_tokens=150, pad_token_id=tokenizer.eos_token_id, do_sample=True, top_k=50, top_p=0.92, temperature=0.75)
    bot_response = tokenizer.decode(new_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    return bot_response, new_history_ids

print('✅ Smart response engine ready!')

✅ Smart response engine ready!


In [5]:
# Test in Terminal
test_msgs = ['Hi, I need help with my order.', 'I want to return a product.', 'My payment failed.', 'How do I track my shipment?']
history = None
for msg in test_msgs:
    print(f'You: {msg}')
    resp, history = get_smart_response(msg, history)
    print(f'Bot: {resp}\n')

You: Hi, I need help with my order.
Bot: I can help with your order! Please share your order number.

You: I want to return a product.
Bot: Happy to help with a return! Our policy allows returns within 30 days. Please share your order number.

You: My payment failed.
Bot: Sorry about the payment issue. Please check your card details or try another payment method.

You: How do I track my shipment?
Bot: To track your shipment, please share your order number.



In [6]:
# Launch Gradio UI
conv_history = {'ids': None}

def chat(user_message, history):
    if not user_message.strip():
        return '', history
    bot_response, conv_history['ids'] = get_smart_response(user_message, conv_history['ids'])
    history.append((user_message, bot_response))
    return '', history

def reset_chat():
    conv_history['ids'] = None
    return [], ''

with gr.Blocks(title='GenAI Customer Support Chatbot') as demo:
    gr.Markdown('# 🤖 GenAI Customer Support Chatbot\n**Powered by DialoGPT + Smart Response Engine**')
    chatbot = gr.Chatbot(label='Customer Support Chat', height=450, bubble_full_width=False)
    with gr.Row():
        user_input = gr.Textbox(placeholder='Type your message...', label='', scale=5)
        send_btn = gr.Button('Send 💬', scale=1, variant='primary')
    clear_btn = gr.Button('🗑️ Clear Chat', variant='secondary')
    gr.Examples(examples=['Hi, I need help with my order.', 'I want to return a product.', 'My payment failed.', 'How do I track my shipment?'], inputs=user_input)
    send_btn.click(chat, [user_input, chatbot], [user_input, chatbot])
    user_input.submit(chat, [user_input, chatbot], [user_input, chatbot])
    clear_btn.click(reset_chat, outputs=[chatbot, user_input])

demo.launch(share=True)

/tmp/ipykernel_12396/3680256596.py:17: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label='Customer Support Chat', height=450, bubble_full_width=False)
/tmp/ipykernel_12396/3680256596.py:17: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(label='Customer Support Chat', height=450, bubble_full_width=False)
/tmp/ipykernel_12396/3680256596.py:17: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label='Customer Support Chat', heig

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc09c24da4a09a98e5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
